# Python de 0 a experto — 7. Pydantic

## Introducción
Pydantic es la librería de validación de datos más usada del ecosistema
Python moderno (es el corazón de FastAPI, y se usa en configuración,
APIs, pipelines de datos...). Retoma justo donde quedó el notebook
anterior: una `@dataclass` organiza datos pero **no valida nada**.
Pydantic sí.

Este es el último notebook de la sesión 2. La siguiente sesión es
Programación Orientada a Objetos (`class_poo/`) — y más adelante SOLID y
Patrones de Diseño. Vale la pena adelantar la conexión: **Pydantic es un
ejemplo real de buen diseño orientado a objetos con validación
declarativa**, algo que se apreciará mucho mejor después de ver SOLID.

## Objetivos
- Explicar qué problema resuelve Pydantic frente a una clase normal o un
  `@dataclass`.
- Definir un `BaseModel` con tipos, validación automática y valores por
  defecto.
- Leer los errores de validación que produce Pydantic (`ValidationError`).
- Reconocer por qué Pydantic es un ejemplo de "buen diseño" (relevante
  para las próximas sesiones de SOLID y Patrones).

## 1. El problema: nada te obliga a validar

Con una clase normal o un `@dataclass`, los type hints son solo
documentación — Python no impide crear un objeto con datos inválidos.

In [1]:
from dataclasses import dataclass

@dataclass
class UsuarioDataclass:
    nombre: str
    edad: int
    email: str

# Esto NO debería ser válido (edad negativa, email sin @, edad como texto)
# pero @dataclass lo permite sin quejarse:
u = UsuarioDataclass(nombre="Ana", edad=-5, email="esto-no-es-un-email")
print(u)
print("edad:", u.edad, "(negativa, y nadie lo impidió)")

u2 = UsuarioDataclass(nombre=123, edad="treinta", email=None)
print(u2, "<- nombre es int, edad es str, email es None. Sigue 'funcionando'.")

UsuarioDataclass(nombre='Ana', edad=-5, email='esto-no-es-un-email')
edad: -5 (negativa, y nadie lo impidió)
UsuarioDataclass(nombre=123, edad='treinta', email=None) <- nombre es int, edad es str, email es None. Sigue 'funcionando'.


## 2. `BaseModel` — la misma idea, pero validada

Se define igual que un `@dataclass` (atributos con anotaciones de tipo),
heredando de `pydantic.BaseModel`. La diferencia aparece al **instanciar**:
Pydantic valida y convierte los datos automáticamente, y rechaza lo que no
puede validar.

In [2]:
from pydantic import BaseModel

class Usuario(BaseModel):
    nombre: str
    edad: int
    email: str

# Pydantic incluso hace coerción razonable de tipos (str numérico -> int)
u = Usuario(nombre="Ana", edad="30", email="ana@example.com")
print(u)
print(type(u.edad), u.edad)   # "30" (str) se convirtió a 30 (int)

nombre='Ana' edad=30 email='ana@example.com'
<class 'int'> 30


In [3]:
from pydantic import ValidationError

# Ahora sí: un dato que no se puede convertir a int falla en la CREACIÓN
# del objeto, no en silencio más adelante en el programa.
try:
    Usuario(nombre="Luis", edad="treinta", email="luis@example.com")
except ValidationError as e:
    print("ValidationError capturado:")
    print(e)

ValidationError capturado:
1 validation error for Usuario
edad
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='treinta', input_type=str]
    For further information visit https://errors.pydantic.dev/2.11/v/int_parsing


## 3. Validación declarativa más fina: `Field` y validadores

Pydantic permite declarar restricciones (rangos, longitudes, formatos)
directamente en la definición del modelo, sin escribir `if` a mano, y
agregar validadores propios cuando la regla es más compleja que un rango.

In [4]:
from pydantic import BaseModel, Field, field_validator

class UsuarioValidado(BaseModel):
    nombre: str = Field(min_length=2, max_length=50)
    edad: int = Field(ge=0, le=120)          # ge=greater/equal, le=less/equal
    email: str = Field(pattern=r".+@.+\..+")  # patrón simple de email

    @field_validator("nombre")
    @classmethod
    def nombre_sin_numeros(cls, valor: str) -> str:
        if any(caracter.isdigit() for caracter in valor):
            raise ValueError("el nombre no puede contener números")
        return valor.title()  # normaliza a "Ana Pérez"


# caso válido
u = UsuarioValidado(nombre="ana perez", edad=30, email="ana@example.com")
print(u)

# tres casos inválidos distintos, todos detectados en la creación
for datos in [
    {"nombre": "Ana2", "edad": 30, "email": "ana@example.com"},   # nombre con número
    {"nombre": "Ana", "edad": 200, "email": "ana@example.com"},   # edad fuera de rango
    {"nombre": "Ana", "edad": 30, "email": "no-es-email"},         # email sin formato válido
]:
    try:
        UsuarioValidado(**datos)
    except ValidationError as e:
        print(f"\nRechazado {datos}:")
        print(e.errors()[0]["msg"])

nombre='Ana Perez' edad=30 email='ana@example.com'

Rechazado {'nombre': 'Ana2', 'edad': 30, 'email': 'ana@example.com'}:
Value error, el nombre no puede contener números

Rechazado {'nombre': 'Ana', 'edad': 200, 'email': 'ana@example.com'}:
Input should be less than or equal to 120

Rechazado {'nombre': 'Ana', 'edad': 30, 'email': 'no-es-email'}:
String should match pattern '.+@.+\..+'


## 4. Serialización: de objeto a dict/JSON y viceversa

Como Pydantic se usa tanto en APIs, viene con conversión incorporada a
`dict` y a JSON en ambas direcciones — algo que con una clase normal
tendrías que escribir a mano.

In [5]:
class Producto(BaseModel):
    nombre: str
    precio: float
    disponible: bool = True

p = Producto(nombre="Teclado mecánico", precio=350000)

print(p.model_dump())          # -> dict de Python
print(p.model_dump_json())     # -> string JSON

# y en la otra dirección: de JSON/dict a objeto validado
json_entrante = '{"nombre": "Mouse", "precio": "89900", "disponible": false}'
p2 = Producto.model_validate_json(json_entrante)
print(p2, "| tipo de precio:", type(p2.precio))

{'nombre': 'Teclado mecánico', 'precio': 350000.0, 'disponible': True}
{"nombre":"Teclado mecánico","precio":350000.0,"disponible":true}
nombre='Mouse' precio=89900.0 disponible=False | tipo de precio: <class 'float'>


## 5. Comparación directa: clase normal vs `@dataclass` vs `BaseModel`

| | Clase normal | `@dataclass` | `BaseModel` (Pydantic) |
|---|---|---|---|
| `__init__` / `__repr__` / `__eq__` | los escribes tú | generados | generados |
| Valida tipos al crear el objeto | no | no | **sí** |
| Convierte tipos compatibles (`"30"` -> `30`) | no | no | **sí** |
| Reglas declarativas (rangos, longitud, regex) | if a mano | if a mano | **`Field(...)`** |
| Serializar a dict/JSON | a mano | `asdict()` (solo dict) | **`.model_dump()` / `.model_dump_json()`** |
| Deserializar validando desde JSON | a mano | a mano | **`.model_validate_json()`** |

## 6. Por qué esto importa para lo que viene

Pydantic no es solo "una librería más": es un ejemplo real de **buen
diseño orientado a objetos**. Cuando veas SOLID en la próxima sesión,
notarás que `BaseModel` aplica ideas muy concretas:
- **Responsabilidad clara**: cada modelo declara sus propias reglas de
  validación, no hay lógica de validación dispersa por el código que lo
  usa.
- **Abierto/cerrado (el "O" de SOLID)**: puedes extender un modelo con
  nuevos validadores sin modificar cómo Pydantic construye el objeto.
- Cuando lleguen los **Patrones de Diseño**, reconocerás que declarar la
  "forma válida de un dato" de una vez (en vez de validar manualmente en
  cada lugar donde se usa) es exactamente el tipo de problema que los
  patrones de diseño buenos resuelven: mover la responsabilidad al lugar
  correcto.

## Ejercicios prácticos

1. Crea un `BaseModel` `Curso` con `nombre: str`, `creditos: int` (entre 1
   y 6) y `activo: bool = True`. Prueba crearlo con datos válidos e
   inválidos y observa el `ValidationError`.
2. Agrega a `UsuarioValidado` un campo `pais: str = "Colombia"` con valor
   por defecto, y crea un usuario sin especificar `pais` para confirmar
   que toma el default.
3. Usa `@field_validator` para validar que `precio` en `Producto` (arriba)
   nunca sea negativo.
4. Convierte un `Producto` a JSON con `.model_dump_json()`, modifica el
   JSON a mano (cambia el precio) y reconstrúyelo con
   `.model_validate_json()`.

## Autoevaluación

- ¿Qué diferencia concreta hay entre pasarle `edad="30"` a un
  `@dataclass` y a un `BaseModel` de Pydantic?
- ¿En qué momento exacto se dispara la validación de Pydantic: al definir
  la clase o al crear una instancia?
- ¿Por qué se dice que Pydantic usa "validación declarativa" en vez de
  "validación imperativa"?

## Referencias
- [Documentación oficial de Pydantic](https://docs.pydantic.dev/latest/)
- [Pydantic — Models](https://docs.pydantic.dev/latest/concepts/models/)
- [Pydantic — Fields](https://docs.pydantic.dev/latest/concepts/fields/)
- [FastAPI — motivación de por qué usa Pydantic](https://fastapi.tiangolo.com/python-types/)